In [1]:
# ---------------- ALBERT / SQuAD-v2  ---------------------------------
import collections, random, numpy as np, torch
import evaluate as eval_lib
from datasets import load_dataset
from tqdm import tqdm
from transformers import AlbertTokenizerFast, AlbertForQuestionAnswering

MODEL_DIR = "../albert_squad2_finetuned/checkpoint-37500"

tokenizer = AlbertTokenizerFast.from_pretrained(MODEL_DIR)
model     = AlbertForQuestionAnswering.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def prepare_features(ex, max_len=384, doc_stride=128):
    pad_right = tokenizer.padding_side == "right"
    tok = tokenizer(
        ex["question" if pad_right else "context"],
        ex["context"  if pad_right else "question"],
        truncation="longest_first", max_length=max_len, stride=doc_stride,
        return_overflowing_tokens=True, return_offsets_mapping=True,
        padding="max_length",
    )
    mapping = tok.pop("overflow_to_sample_mapping")
    tok["example_id"] = [ex["id"][i] for i in mapping]
    return tok

def postprocess(preds, feats, exs):
    s_log, e_log = preds
    per_ex = collections.defaultdict(list)
    for i, ex_id in enumerate(feats["example_id"]): per_ex[ex_id].append(i)
    final, n_best, max_len = collections.OrderedDict(), 20, 30
    for ex in exs:
        cand = []
        for fi in per_ex[ex["id"]]:
            offs = feats["offset_mapping"][fi]
            for s in np.argsort(s_log[fi])[-n_best:]:
                for e in np.argsort(e_log[fi])[-n_best:]:
                    if e < s or e - s + 1 > max_len or offs[s] is None or offs[e] is None:
                        continue
                    cand.append({"score": s_log[fi][s] + e_log[fi][e],
                                 "start": offs[s][0], "end": offs[e][1]})
        if cand:
            best = max(cand, key=lambda x: x["score"])
            final[ex["id"]] = {"text": ex["context"][best["start"]:best["end"]],
                               "score": best["score"]}
        else:
            final[ex["id"]] = {"text":"", "score":0.0}
    return final

# ---------- slow forward-pass ----------------------------------------
print("\nLoading SQuAD-v2 validation …")
examples = load_dataset("squad_v2")["validation"]
print(f"Tokenising {len(examples)} examples …")
features = prepare_features(examples)

all_start, all_end, bs = [], [], 8
print(f"Running model on {len(features['input_ids'])} features (bs={bs}) …")
for i in tqdm(range(0, len(features["input_ids"]), bs), desc="Predicting"):
    batch = {k: torch.tensor(v[i:i+bs]).to(device)
             for k,v in features.items() if k in ["input_ids","attention_mask"]}
    with torch.no_grad():
        out = model(**batch)
    all_start.extend(out.start_logits.cpu().numpy())
    all_end.extend(out.end_logits.cpu().numpy())

print("Inference done → proceed to Cell 2.")


/Users/seanhall/Desktop/NLPFinalProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading SQuAD-v2 validation …
Tokenising 11873 examples …
Running model on 12171 features (bs=8) …


Predicting: 100%|██████████| 1522/1522 [31:11<00:00,  1.23s/it]

Inference done → proceed to Cell 2.


In [2]:
# ---- ALBERT post-processing + metrics --------------------------------
print("\nPost-processing …")
preds = postprocess((all_start, all_end), features, examples)

metric   = eval_lib.load("squad_v2")
refs     = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
predlist = [{"id": ex["id"],
             "prediction_text": preds[ex["id"]]["text"],
             "no_answer_probability": 1.0 if preds[ex["id"]]["text"]=="" else 0.0}
            for ex in examples]

res   = metric.compute(predictions=predlist, references=refs)
print(f"\nEM: {res['exact']:.2f} | F1: {res['f1']:.2f}")

wrong = [ {"q":ex["question"], "p":preds[ex["id"]]["text"],
           "g":ex["answers"]["text"][0] if ex["answers"]["text"] else "No answer"}
          for ex in examples
          if (ex['answers']['text'] and preds[ex['id']]['text'] not in ex['answers']['text'])
          or (not ex['answers']['text'] and preds[ex['id']]['text']!="") ]

for i,s in enumerate(random.sample(wrong, min(3,len(wrong))),1):
    print(f"\nEx {i}\nQ: {s['q']}\nPred: '{s['p']}'\nGold: '{s['g']}'")



Post-processing …

EM: 75.67 | F1: 79.26

Ex 1
Q: When did the the German army re-occupy Britain and France?
Pred: '1936'
Gold: 'No answer'

Ex 2
Q: What is the name of Harvard's primary recreational sports facility?
Pred: 'The Malkin Athletic Center'
Gold: 'Malkin Athletic Center'

Ex 3
Q: Who separated a number of earlier theories into a set of 20 scalar equations?
Pred: 'James Clerk Maxwell'
Gold: 'No answer'
